In [2]:
import asyncio
import httpx
import pandas as pd
import nltk
from tqdm import tqdm

nltk.download('punkt_tab', quiet=True)

True

In [ ]:


GATEWAY_URL = "http://localhost:8009/chat"
K = 5  # número de votos
CONCURRENCY_LIMIT = 15  # evita sobrecarga

semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)


# -------------------------
# 🔥 chamada ao gateway
# -------------------------

async def call_model(client, prompt):
    async with semaphore:
        try:
            resp = await client.post(
                GATEWAY_URL,
                json={"prompt": prompt},
                timeout=300
            )
            
            # Verificar status HTTP
            if resp.status_code != 200:
                print(f"\n{'='*80}")
                print(f"❌ ERRO HTTP {resp.status_code}")
                print(f"{'='*80}")
                print(f"Prompt enviado (primeiros 500 chars):")
                print(f"{prompt[:500]}...")
                print(f"\nResposta completa:")
                print(resp.text)
                print(f"\nHeaders:")
                print(dict(resp.headers))
                print(f"{'='*80}\n")
                return ""
            
            # Tentar fazer parse do JSON
            try:
                data = resp.json()
            except Exception as json_err:
                print(f"\n{'='*80}")
                print(f"❌ Erro ao parsear JSON: {json_err}")
                print(f"{'='*80}")
                print(f"Resposta recebida:")
                print(resp.text)
                print(f"{'='*80}\n")
                return ""
            
            # Verificar estrutura da resposta
            if "choices" not in data or len(data["choices"]) == 0:
                print(f"\n{'='*80}")
                print(f"❌ Resposta sem 'choices'")
                print(f"{'='*80}")
                print(f"Dados recebidos:")
                print(data)
                print(f"{'='*80}\n")
                return ""
            
            return data["choices"][0]["message"]["content"]
            
        except httpx.TimeoutException:
            print(f"❌ Timeout na requisição (>300s)")
            return ""
        except Exception as e:
            print(f"\n{'='*80}")
            print(f"❌ Erro inesperado: {type(e).__name__}")
            print(f"{'='*80}")
            print(f"Mensagem: {str(e)}")
            import traceback
            print(f"\nTraceback:")
            traceback.print_exc()
            print(f"{'='*80}\n")
            return ""


# -------------------------
# 🔥 avalia uma sentença (K votos em paralelo)
# -------------------------
async def avaliar_sentenca(client, contexto, sent):
    prompt = f"Context: {contexto}\n\nStatement: {sent}\n\nIs the statement faithful to the context?"

    tasks = [
        call_model(client, prompt)
        for _ in range(K)
    ]

    respostas = await asyncio.gather(*tasks)

    votos_unfaithful = 0

    for texto in respostas:
        texto = texto.lower()
        if "unfaithful" in texto or "hallucination" in texto:
            votos_unfaithful += 1

    return votos_unfaithful / K


# -------------------------
# 🔥 processa um relatório inteiro
# -------------------------
async def processar_relatorio(client, row):
    contexto = row['contexto_completo']
    relatorio = row['relatorio_ia']
    sentencas = nltk.sent_tokenize(relatorio)

    tasks = [
        avaliar_sentenca(client, contexto, sent)
        for sent in sentencas
    ]

    scores = await asyncio.gather(*tasks)

    resultados = []
    for sent, score in zip(sentencas, scores):
        resultados.append({
            "id_bo": row['codigo_bo'],
            "sentenca": sent,
            "non_conformity_score": score
        })

    return resultados


# -------------------------
# 🔥 pipeline principal
# -------------------------
async def auditar(csv_relatorios):
    df = pd.read_csv(csv_relatorios)

    print(f"Iniciando auditoria de {len(df)} relatórios...")

    async with httpx.AsyncClient() as client:
        tasks = [
            processar_relatorio(client, row)
            for _, row in df.iterrows()
        ]

        resultados = []
        
        for coro in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            res = await coro
            resultados.extend(res)

    # salvar CSV final
    df_final = pd.DataFrame(resultados)
    df_final.to_csv("sentencas_auditadas_votos.csv", index=False)

    print("\n✅ Auditoria concluída!")
    print("Arquivo gerado: sentencas_auditadas_votos.csv")


In [4]:
#asyncio.run(auditar("dataset_com_relatorios.csv"))

# No Jupyter, use:
await auditar("dataset_com_relatorios.csv")

Iniciando auditoria de 100 relatórios...


100%|██████████| 100/100 [14:32<00:00,  8.73s/it]


✅ Auditoria concluída!
Arquivo gerado: sentencas_auditadas_votos.csv
